# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%204/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before I lock in a rule, I check the signals it would lean on — the session showed a rule is only as honest as the signals behind it. I test two candidates against the real trailing-90-day data: staleness (the signal behind FlyRank's refresh-tier flags) and CTR-vs-position (the signal behind the `needs_ctr_fix` flag). Whichever one holds up cleanly is the one I build the rule on.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Trailing-90-day label, built the same way notebook 02 does it. Never used as a scoring input below.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | base decline rate {df['is_declining_label'].mean():.3f}")

Note: you may need to restart the kernel to use updated packages.
Working dir: /tmp/flyrank-ml-internship


30,000 pages | 32 clients | base decline rate 0.542


### Signal 1 — staleness (behind FlyRank's refresh-tier flags)

My first candidate: *pages get more likely to be declining the longer it's been since their last update.* That's the assumption FlyRank's refresh-tier flags run on. I bucket every page by days since its last update and check whether decline rate actually climbs with staleness.

In [2]:
bins = [-1, 30, 90, 180, 365, np.inf]
tier_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["freshness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=tier_labels)

staleness_table = (
    df.groupby("freshness_bucket", observed=True)["is_declining_label"]
    .agg(n="count", decline_rate="mean")
    .reindex(tier_labels)
)
print("Signal 1 -- staleness vs. decline rate")
print(staleness_table.round(3))
print("\nVerdict: MIXED")

Signal 1 -- staleness vs. decline rate
                      n  decline_rate
freshness_bucket                     
0-30              20480         0.511
31-90               175         0.589
91-180             9171         0.611
181-365             169         0.467
365+                  5         0.600

Verdict: MIXED


**Verdict: MIXED.** Decline rate doesn't climb steadily with staleness — it rises through the 91-180 bucket, then drops back down at 181-365, and the two oldest buckets (31-90 and 365+) barely have any pages in them at all (n=175 and n=5) to trust. If I built my rule on "older = more likely declining" the way FlyRank's refresh-tier flags assume, I'd be reading noise as signal on this slice. That's a real result, not a rule I get to keep — staleness doesn't earn a place in my score.

### Signal 2 — CTR vs. position (behind FlyRank's `needs_ctr_fix` flag)

Second candidate: *pages ranking well but getting fewer clicks than their position should earn are worth reviewing.* That's the logic behind FlyRank's CTR-fix flag. I bucket every page with real position data by position tier and check whether CTR actually falls as position gets worse.

In [3]:
pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
has_pos = df["avg_position"] > 0
df.loc[has_pos, "position_tier"] = pd.cut(df.loc[has_pos, "avg_position"], bins=pos_bins, labels=pos_labels)

ctr_table = (
    df[has_pos]
    .groupby("position_tier", observed=True)
    .agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"), decline_rate=("is_declining_label", "mean"))
    .reindex(pos_labels)
)
print(f"Signal 2 -- CTR by position tier (n with real position data: {has_pos.sum():,} / {len(df):,})")
print(ctr_table.round(3))
print("\nVerdict: CONFIRMED")

Signal 2 -- CTR by position tier (n with real position data: 28,795 / 30,000)
                   n  mean_ctr  decline_rate
position_tier                               
top_3           1141     2.714         0.498
page_1         11842     0.651         0.569
striking        7273     0.323         0.610
page_3_5        7225     0.222         0.562
deep            1314     0.151         0.343

Verdict: CONFIRMED


**Verdict: CONFIRMED.** Mean CTR falls in a straight line as position tier gets worse — 2.71% at top_3, down to 0.65%, 0.32%, 0.22%, then 0.15% at deep — with solid n at every tier (over a thousand pages even in the smallest bucket). This is the clean signal, and it's the one behind FlyRank's `needs_ctr_fix` flag. I build my rule on this one.

### The rule, stated plainly

**A page is worth a CTR review if it holds a strong position, gets real traffic, and its click-through rate is falling short of what pages at that same position tier typically earn.** I score it as how far below the tier's average CTR it sits, times how much traffic is riding on it — so pages that are both underperforming and high-volume rank first.

Every flagged page carries the same single reason code, `low_ctr_for_position` — this rule only ever explains itself one way. The action label is `review_ctr`; pages that don't clear the bar get `no_action` and no score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I encode the rule from Section 1 directly — no fitted weights, just the three trailing-90-day inputs (position, CTR, impressions) multiplied together.

In [4]:
tier_mean_ctr = df[has_pos].groupby("position_tier", observed=True)["ctr"].mean()
df["expected_ctr"] = df["position_tier"].map(tier_mean_ctr)

visible = (df["impressions_90d"] >= 100).astype(int)                             # real demand, not a fluke of one crawl
reachable = ((df["avg_position"] > 0) & (df["avg_position"] <= 50)).astype(int)  # has real position data and is close enough to matter
ctr_gap = (df["expected_ctr"] - df["ctr"]).clip(lower=0).fillna(0)               # how far below its tier's typical CTR

df["score"] = visible * reachable * ctr_gap * np.log1p(df["impressions_90d"])
df["reason_code"] = np.where(df["score"] > 0, "low_ctr_for_position", "no_flag")
df["action"] = np.where(df["score"] > 0, "review_ctr", "no_action")
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

flagged = df[df["score"] > 0]
print(f"{len(flagged):,} / {len(df):,} pages flagged for CTR review")
print(f"decline rate -- flagged: {flagged['is_declining_label'].mean():.3f}  |  base rate: {df['is_declining_label'].mean():.3f}")

out_cols = [
    "content_id", "client_id", "rank", "score", "reason_code", "action",
    "position_tier", "avg_position", "impressions_90d", "ctr", "expected_ctr",
    "is_declining_label",
]
out = df.sort_values("rank")[out_cols]

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
out.to_csv(out_path, index=False)
print(f"Wrote {len(out):,} ranked rows to {out_path}")

17,058 / 30,000 pages flagged for CTR review
decline rate -- flagged: 0.629  |  base rate: 0.542


Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of my top 10: action, reason code, and what would make it wrong.*

(The assignment card for this round asks for a top-10 review, not the skeleton's default top-20 — I'm following the card.)

In [5]:
top10 = out.head(10)
print(top10.to_string(index=False))

          content_id         client_id  rank     score          reason_code     action position_tier  avg_position  impressions_90d  ctr  expected_ctr  is_declining_label
content_8c19996aa890 client_4e07408562     1 33.696740 low_ctr_for_position review_ctr         top_3           2.5           509252 0.15      2.714303                   1
content_8451fc6f034d client_d029fa3a95     2 33.591613 low_ctr_for_position review_ctr         top_3           2.3           272144 0.03      2.714303                   0
content_4a6607efcb46 client_6208ef0f77     3 31.803484 low_ctr_for_position review_ctr         top_3           2.2           128068 0.01      2.714303                   0
content_e12868d1f396 client_4e07408562     4 31.510775 low_ctr_for_position review_ctr         top_3           2.9           149712 0.07      2.714303                   0
content_4c36c775b818 client_4e07408562     5 30.061265 low_ctr_for_position review_ctr         top_3           2.3           463103 0.41      2.7

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# Leakage check -- the score and reason code above are built ONLY from avg_position, ctr,
# and impressions_90d, all trailing-90-day inputs from the same window. Confirm nothing
# label-derived or product-flag-derived ever touched the score.
scoring_inputs = {"avg_position", "ctr", "impressions_90d"}
forbidden = {
    "trend_direction", "trend_pct", "is_declining_label",   # label-derived
    "health_score", "needs_ctr_fix", "is_quick_win",        # FlyRank product flags
    "priority_score", "action_type", "refresh_tier",
}
assert scoring_inputs.isdisjoint(forbidden), "a forbidden column leaked into the score inputs"
shipped_flags = (forbidden & set(df.columns)) - {"trend_direction", "trend_pct", "is_declining_label"}
print("Scoring inputs used:", sorted(scoring_inputs))
print("FlyRank product flags present in this CSV at all:", sorted(shipped_flags) or "none -- they aren't shipped to interns")
print("Leakage check passed: no future window, no label-derived column, no product flag fed the score.")

# Weak-pick evidence
top50 = out.head(50)
print(f"\ntop-50 decline rate: {top50['is_declining_label'].mean():.3f}  (base rate: {df['is_declining_label'].mean():.3f})")
print("top-50 tier mix:")
print(top50["position_tier"].value_counts())
print("\ntop-50 client concentration (top 5 clients):")
print(top50["client_id"].value_counts().head(5))

Scoring inputs used: ['avg_position', 'ctr', 'impressions_90d']
FlyRank product flags present in this CSV at all: none -- they aren't shipped to interns
Leakage check passed: no future window, no label-derived column, no product flag fed the score.

top-50 decline rate: 0.680  (base rate: 0.542)
top-50 tier mix:
position_tier
top_3    50
Name: count, dtype: int64

top-50 client concentration (top 5 clients):
client_id
client_19581e27de    25
client_4e07408562    10
client_f369cb89fc     6
client_6208ef0f77     2
client_7f2253d7e2     2
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `client_id`/`content_id` values, same as the rest of this repo
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

The last box is mine to check after I run **File → Save a copy in GitHub** from Colab (or push from my own clone) — this environment can't push to my repo directly.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.